# Text Audit and Minimal Preprocessing

This notebook documents the quality audit and non-destructive text preparation applied after LSTM silver labelling and before group-aware partitioning. It preserves the original review text, creates a lightly cleaned representation, and creates a lowercase representation for lexical modelling.


## Release Boundary

This is a restricted code-inspection copy. It retains the original analytical logic, but it does not include Banjarbaru source data, review-level records, manual-audit files, trained checkpoints, or intermediate outputs. The paths in the code refer to the authorized analysis environment and are not expected to run from this public package.


### Install audit dependencies

Installs the WordCloud and emoji dependencies used in the historical audit environment.


In [ ]:
!pip install wordcloud emoji -q
print("Library berhasil diinstall!")


### Import audit libraries and set the random state

Imports data-processing, Unicode, plotting, and emoji utilities, then fixes the historical random seed.


In [ ]:
import pandas as pd
import numpy as np
import re
import os
import unicodedata
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import emoji

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Library berhasil diimport!")
print(f"Pandas version: {pd.__version__}")


### Define restricted input and output locations

Declares the historical silver-labelled input, processed-data output, and figure locations. These paths refer to the restricted analysis environment.


In [ ]:
INPUT_PATH = "../data/interim/labeled_reviews_lstm.csv"
OUTPUT_PATH = "../data/processed/preprocessed_reviews.csv"
FIGURES_PATH = "../figures/"

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
os.makedirs(FIGURES_PATH, exist_ok=True)

print(f"Input: {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")
print(f"Figures: {FIGURES_PATH}")


### Load the silver-labelled review dataset

Reads the restricted LSTM-labelled dataset and checks its row count, column names, and data types.


In [ ]:
df = pd.read_csv(INPUT_PATH)

print("="*60)
print("INFORMASI DATASET")
print("="*60)
print(f"\nJumlah baris: {len(df):,}")
print(f"Jumlah kolom: {len(df.columns)}")
print(f"\nKolom yang tersedia:")
for col in df.columns:
    print(f"  - {col}: {df[col].dtype}")


### Inspect representative local records

Displays selected review, silver-label, and confidence fields locally to validate the expected input structure. No records are released.


In [ ]:
print("\nSampel data:")
df[['text_raw', 'sentiment_label', 'confidence']].head(10)


### Check missing values

Calculates missing-value counts and percentages for every input column before preprocessing.


In [ ]:
print("\nMissing values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing': missing,
    'Percentage': missing_pct
})
print(missing_df[missing_df['Missing'] > 0])
if missing_df['Missing'].sum() == 0:
    print("Tidak ada missing values!")


### Summarize the silver-label distribution

Reports the class counts, class percentages, and max-to-min imbalance ratio inherited from LSTM silver labelling.


In [ ]:
print("="*60)
print("DISTRIBUSI KELAS SENTIMEN")
print("="*60)

label_counts = df['sentiment_label'].value_counts()
label_pct = df['sentiment_label'].value_counts(normalize=True) * 100

print(f"\nDistribusi kelas:")
for label in ['positif', 'netral', 'negatif']:
    if label in label_counts.index:
        print(f"  {label.capitalize():10s}: {label_counts[label]:,} ({label_pct[label]:.2f}%)")

max_class = label_counts.max()
min_class = label_counts.min()
imbalance_ratio = max_class / min_class

print(f"\nRasio imbalance (max/min): {imbalance_ratio:.2f}:1")
if imbalance_ratio > 3:
    print("WARNING: Class imbalance > 3:1")


### Calculate review-length statistics

Adds character and whitespace-token counts, then reports their descriptive statistics across the silver-labelled dataset.


In [ ]:
df['char_len'] = df['text_raw'].str.len()
df['word_count'] = df['text_raw'].str.split().str.len()

print("="*60)
print("STATISTIK PANJANG TEKS")
print("="*60)

print(f"\nPanjang Karakter:")
print(f"  Mean   : {df['char_len'].mean():.1f}")
print(f"  Median : {df['char_len'].median():.1f}")
print(f"  Std    : {df['char_len'].std():.1f}")
print(f"  Min    : {df['char_len'].min()}")
print(f"  Max    : {df['char_len'].max()}")
print(f"  Q1     : {df['char_len'].quantile(0.25):.1f}")
print(f"  Q3     : {df['char_len'].quantile(0.75):.1f}")

print(f"\nJumlah Kata:")
print(f"  Mean   : {df['word_count'].mean():.1f}")
print(f"  Median : {df['word_count'].median():.1f}")
print(f"  Std    : {df['word_count'].std():.1f}")
print(f"  Min    : {df['word_count'].min()}")
print(f"  Max    : {df['word_count'].max()}")


### Quantify short reviews

Measures the proportion of reviews with ten or fewer words, overall and within each silver-label class.


In [ ]:
short_texts = df[df['word_count'] <= 10]
short_pct = len(short_texts) / len(df) * 100

print(f"\nProporsi ulasan pendek (<=10 kata): {len(short_texts):,} ({short_pct:.1f}%)")

print(f"\nDistribusi teks pendek per kelas:")
for label in ['positif', 'netral', 'negatif']:
    total_label = len(df[df['sentiment_label'] == label])
    short_label = len(short_texts[short_texts['sentiment_label'] == label])
    pct = short_label / total_label * 100 if total_label > 0 else 0
    print(f"  {label.capitalize():10s}: {short_label:,} dari {total_label:,} ({pct:.1f}%)")


### Extract text-style indicators

Creates indicators for emoji, URLs, mentions, expressive punctuation, character elongation, uppercase tokens, and Indonesian negation words.


In [ ]:
def count_emoji(text):
    """Hitung jumlah emoji dalam teks"""
    return len([c for c in text if c in emoji.EMOJI_DATA])

def has_url(text):
    """Cek apakah teks mengandung URL"""
    return bool(re.search(r'https?://\S+|www\.\S+', text))

def has_mention(text):
    """Cek apakah teks mengandung mention @user"""
    return bool(re.search(r'@\w+', text))

def count_exclamation(text):
    """Hitung jumlah tanda seru"""
    return text.count('!')

def count_question(text):
    """Hitung jumlah tanda tanya"""
    return text.count('?')

def has_elongation(text):
    """Cek apakah teks mengandung elongation (misal: enaaak, baguuus)"""
    return bool(re.search(r'(.)\1{2,}', text))

def has_uppercase(text):
    """Cek apakah teks mengandung kata dengan HURUF BESAR"""
    return bool(re.search(r'[A-Z]{3,}', text))

def has_negation(text):
    """Cek apakah teks mengandung kata negasi"""
    negation_words = ['tidak', 'nggak', 'ga', 'gak', 'bukan', 'kurang', 'belum', 'jangan', 'tanpa']
    text_lower = text.lower()
    return any(word in text_lower.split() for word in negation_words)

print("Menghitung fitur karakteristik teks...")
df['emoji_count'] = df['text_raw'].apply(count_emoji)
df['has_emoji'] = df['emoji_count'] > 0
df['has_url'] = df['text_raw'].apply(has_url)
df['has_mention'] = df['text_raw'].apply(has_mention)
df['exclamation_count'] = df['text_raw'].apply(count_exclamation)
df['question_count'] = df['text_raw'].apply(count_question)
df['has_elongation'] = df['text_raw'].apply(has_elongation)
df['has_uppercase'] = df['text_raw'].apply(has_uppercase)
df['has_negation'] = df['text_raw'].apply(has_negation)

print("Selesai!")


### Report text-style indicator frequencies

Summarizes how often each style indicator and expressive punctuation feature appears in the review corpus.


In [ ]:
print("="*60)
print("STATISTIK KARAKTERISTIK TEKS")
print("="*60)

features = [
    ('Mengandung Emoji', 'has_emoji'),
    ('Mengandung URL', 'has_url'),
    ('Mengandung Mention', 'has_mention'),
    ('Mengandung Elongation', 'has_elongation'),
    ('Mengandung UPPERCASE', 'has_uppercase'),
    ('Mengandung Negasi', 'has_negation'),
]

print(f"\n{'Fitur':<25} {'Jumlah':>10} {'Persentase':>12}")
print("-"*50)
for name, col in features:
    count = df[col].sum()
    pct = count / len(df) * 100
    print(f"{name:<25} {count:>10,} {pct:>11.2f}%")

print(f"\n\nTanda Baca Ekspresif:")
print(f"  Ulasan dengan '!': {(df['exclamation_count'] > 0).sum():,} ({(df['exclamation_count'] > 0).sum()/len(df)*100:.2f}%)")
print(f"  Ulasan dengan '?': {(df['question_count'] > 0).sum():,} ({(df['question_count'] > 0).sum()/len(df)*100:.2f}%)")
print(f"  Rata-rata '!' per ulasan: {df['exclamation_count'].mean():.2f}")
print(f"  Rata-rata '?' per ulasan: {df['question_count'].mean():.2f}")


### Inspect local edge-case examples

Displays local examples containing negation, elongation, uppercase text, and very short reviews to support preprocessing review. No examples are released.


In [ ]:
print("="*60)
print("CONTOH EDGE CASES")
print("="*60)

print("\nUlasan dengan NEGASI (5 sampel):")
negation_samples = df[df['has_negation']][['text_raw', 'sentiment_label']].head(5)
for idx, row in negation_samples.iterrows():
    text = row['text_raw'][:100] + '...' if len(row['text_raw']) > 100 else row['text_raw']
    print(f"  [{row['sentiment_label'].upper()}] {text}")

print("\nUlasan dengan ELONGATION (5 sampel):")
elong_samples = df[df['has_elongation']][['text_raw', 'sentiment_label']].head(5)
for idx, row in elong_samples.iterrows():
    text = row['text_raw'][:100] + '...' if len(row['text_raw']) > 100 else row['text_raw']
    print(f"  [{row['sentiment_label'].upper()}] {text}")

print("\nUlasan dengan UPPERCASE (5 sampel):")
upper_samples = df[df['has_uppercase']][['text_raw', 'sentiment_label']].head(5)
for idx, row in upper_samples.iterrows():
    text = row['text_raw'][:100] + '...' if len(row['text_raw']) > 100 else row['text_raw']
    print(f"  [{row['sentiment_label'].upper()}] {text}")

print("\nUlasan SANGAT PENDEK (<=5 kata, 5 sampel):")
short_samples = df[df['word_count'] <= 5][['text_raw', 'sentiment_label', 'word_count']].head(5)
for idx, row in short_samples.iterrows():
    print(f"  [{row['sentiment_label'].upper()}] ({row['word_count']} kata) {row['text_raw']}")


### Define the minimal preprocessing policy

Normalizes Unicode, removes URLs and mentions, collapses repeated whitespace, and preserves case, emojis, punctuation, and other review content.


In [ ]:
def preprocess_text(text):
    
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    text = unicodedata.normalize('NFKC', text)
    
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    text = re.sub(r'@\w+', '', text)
    
    text = re.sub(r'\s+', ' ', text)
    
    text = text.strip()
    
    return text

def create_lowercase_version(text):

    return text.lower() if isinstance(text, str) else ""

print("Fungsi pra-pemrosesan didefinisikan!")


### Create retained text representations

Copies raw text to text_original, applies minimal cleaning to create text_clean, and creates text_lower for the lexical model input.


In [ ]:
print("Menerapkan pra-pemrosesan...")

df['text_original'] = df['text_raw'].copy()

df['text_clean'] = df['text_raw'].apply(preprocess_text)

df['text_lower'] = df['text_clean'].apply(create_lowercase_version)

print("Selesai!")
print(f"\nKolom yang tersedia sekarang:")
print(f"  - text_original: Teks asli sebelum preprocessing")
print(f"  - text_clean: Teks setelah preprocessing ringan (pertahankan case)")
print(f"  - text_lower: Versi lowercase untuk TF-IDF")


### Inspect preprocessing changes locally

Counts modified reviews and displays local before-and-after examples to verify that the preprocessing policy has the intended scope.


In [ ]:
print("="*60)
print("PERBANDINGAN SEBELUM DAN SESUDAH PREPROCESSING")
print("="*60)

changed_mask = df['text_original'] != df['text_clean']
changed_samples = df[changed_mask][['text_original', 'text_clean']].head(5)

print(f"\nJumlah teks yang berubah: {changed_mask.sum():,} dari {len(df):,} ({changed_mask.sum()/len(df)*100:.2f}%)")

if len(changed_samples) > 0:
    print("\nContoh perubahan:")
    for idx, row in changed_samples.iterrows():
        before = row['text_original'][:80] + '...' if len(row['text_original']) > 80 else row['text_original']
        after = row['text_clean'][:80] + '...' if len(row['text_clean']) > 80 else row['text_clean']
        print(f"\n  BEFORE: {before}")
        print(f"  AFTER : {after}")


### Verify preservation and removal rules

Checks that emojis, exclamation marks, and uppercase tokens are retained while URLs are removed, then discards temporary validation columns.


In [ ]:
print("="*60)
print("VALIDASI PRESERVASI ELEMEN PENTING")
print("="*60)

df['emoji_count_after'] = df['text_clean'].apply(count_emoji)
emoji_preserved = (df['emoji_count'] == df['emoji_count_after']).all()
print(f"\nEmoji dipertahankan: {emoji_preserved}")

df['excl_after'] = df['text_clean'].apply(count_exclamation)
excl_preserved = (df['exclamation_count'] == df['excl_after']).all()
print(f"Tanda seru (!) dipertahankan: {excl_preserved}")

df['upper_after'] = df['text_clean'].apply(has_uppercase)
upper_preserved = (df['has_uppercase'] == df['upper_after']).all()
print(f"UPPERCASE dipertahankan: {upper_preserved}")

df['url_after'] = df['text_clean'].apply(has_url)
url_removed = (df['url_after'].sum() == 0)
print(f"URL dihapus: {url_removed}")

df.drop(columns=['emoji_count_after', 'excl_after', 'upper_after', 'url_after'], inplace=True)


### Summarize frequent within-class bigrams

Uses CountVectorizer to list frequent bigrams within each silver-label class as a descriptive lexical audit.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def get_top_ngrams(corpus, n=2, top_k=15):
    """Dapatkan n-gram paling sering"""
    vec = CountVectorizer(ngram_range=(n, n), stop_words=None).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return words_freq[:top_k]

print("="*60)
print("TOP 15 BIGRAM PER KELAS SENTIMEN")
print("="*60)

for label in ['positif', 'netral', 'negatif']:
    corpus = df[df['sentiment_label'] == label]['text_lower'].tolist()
    top_bigrams = get_top_ngrams(corpus, n=2, top_k=15)
    
    print(f"\n{label.upper()}:")
    for i, (ngram, freq) in enumerate(top_bigrams, 1):
        print(f"  {i:2d}. '{ngram}' - {freq:,}x")


### Summarize inherited LSTM confidence

Reports the overall and class-specific LSTM confidence distribution and the frequency below a 0.70 threshold.


In [ ]:
print("="*60)
print("STATISTIK CONFIDENCE SCORE")
print("="*60)

print(f"\nKeseluruhan:")
print(f"  Mean   : {df['confidence'].mean():.4f}")
print(f"  Median : {df['confidence'].median():.4f}")
print(f"  Std    : {df['confidence'].std():.4f}")
print(f"  Min    : {df['confidence'].min():.4f}")
print(f"  Max    : {df['confidence'].max():.4f}")

print(f"\nPer Kelas Sentimen:")
for label in ['positif', 'netral', 'negatif']:
    subset = df[df['sentiment_label'] == label]
    print(f"  {label.capitalize():10s}: Mean={subset['confidence'].mean():.4f}, Median={subset['confidence'].median():.4f}")

low_conf = df[df['confidence'] < 0.7]
print(f"\nUlasan dengan confidence < 0.7: {len(low_conf):,} ({len(low_conf)/len(df)*100:.2f}%)")


### Define the processed-data schema

Selects the identifiers, text representations, lengths, silver labels, probabilities, confidence, and retained style indicators needed by later private stages.


In [ ]:
columns_to_keep = [
    'Place',
    'Actual Place Name',
    'Place Type',
    'text_original',
    'text_clean',
    'text_lower',
    'word_count',
    'char_len',
    'sentiment_label',
    'confidence',
    'prob_positif',
    'prob_netral',
    'prob_negatif',
    'has_emoji',
    'has_negation',
    'has_elongation',
    'has_uppercase',
    'exclamation_count',
    'question_count'
]

available_columns = [col for col in columns_to_keep if col in df.columns]
df_output = df[available_columns].copy()

print(f"Kolom yang akan disimpan ({len(available_columns)}):")
for col in available_columns:
    print(f"  - {col}")


### Write the processed review dataset

Writes the restricted processed dataset used for partitioning and model development. The row-level file is excluded.


In [ ]:
df_output.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')

print("="*60)
print("OUTPUT TERSIMPAN")
print("="*60)
print(f"\nDataset: {OUTPUT_PATH}")
print(f"  Jumlah baris: {len(df_output):,}")
print(f"  Jumlah kolom: {len(df_output.columns)}")
print(f"  Ukuran file: {os.path.getsize(OUTPUT_PATH) / 1024 / 1024:.2f} MB")
